In [1]:
import pandas as pd
import requests
from PIL import Image
from io import BytesIO
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

In [2]:
airbnb = pd.read_csv('Airbnb_NYC.csv')

In [ ]:
def load_image_from_url(url):
    try:
        response = requests.get(url, timeout=3.5)
        response.raise_for_status()
        img = Image.open(BytesIO(response.content))
        return img
    except Exception as e:
        return None

def load_images_parallel(urls, max_workers=10):
    results = [None] * len(urls)
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_idx = {executor.submit(load_image_from_url, url): i 
                         for i, url in enumerate(urls)}
        with tqdm(total=len(urls), desc="Downloading images") as pbar:
            for future in as_completed(future_to_idx):
                idx = future_to_idx[future]
                results[idx] = future.result()
                pbar.update(1)
    return results

images = load_images_parallel(airbnb['picture_url'].tolist())

valid_mask = [img is not None for img in images]
airbnb_clean = airbnb[valid_mask].copy()
images_clean = [img for img in images if img is not None]

print(f"Started with {len(airbnb)} rows")
print(f"Kept {len(airbnb_clean)} rows")
print(f"Removed {len(airbnb) - len(airbnb_clean)} rows")

In [ ]:
#AirBnB Dataframe without broken links

airbnb_clean

In [ ]:
# Images list

images_clean

In [ ]:
# Example Image

images_clean[0]

In [ ]:
images_clean = pd.DataFrame(images_clean, columns=["Images"])

In [ ]:
# Make new dataframe downloadable 

airbnb_clean.to_csv("airbnb_clean.csv", index=False)

In [ ]:
# Make new dataframe downloadable 

images_clean.to_csv("images_clean.csv", index=False)